# Hathaway (2015) — "The Solar Cycle" Implementation / 태양 활동 주기 구현

**Paper**: Hathaway, D. H., *Living Reviews in Solar Physics*, **12**, 4 (2015), DOI: 10.1007/lrsp-2015-4

## Purpose / 목적

**English.** This notebook reproduces the key empirical laws of the solar cycle reviewed by Hathaway (2015) using synthetic sunspot data. We implement and visualize:
1. Synthetic sunspot number time series (Schwabe 11-year cycle with amplitude modulation + noise)
2. Butterfly diagram (Spörer's Law latitude drift)
3. Waldmeier effect (rise time vs amplitude)
4. Power spectrum (Schwabe + Gleissberg modulation)
5. Polar-field precursor method for cycle prediction
6. Simple Babcock–Leighton-style flux transport dynamo toy model

**한국어.** 이 노트북은 Hathaway(2015)가 리뷰한 태양 주기의 주요 경험 법칙을 합성(synthetic) 흑점 자료로 재현한다. 구현 및 시각화 항목:
1. 합성 흑점수 시계열 (Schwabe 11년 주기 + 진폭 변조 + 잡음)
2. Butterfly diagram (Spörer 법칙의 위도 이동)
3. Waldmeier 효과 (상승시간 vs 진폭)
4. 파워 스펙트럼 (Schwabe + Gleissberg 변조)
5. 극자기장 전조 주기 예측 방법
6. 단순 Babcock–Leighton 스타일 flux transport dynamo 장난감 모델

## Kernel / 커널
`study-with-ai` conda environment.

In [ ]:
# Imports / 임포트
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.optimize import curve_fit

np.random.seed(43)  # paper_id = 43

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("NumPy:", np.__version__)
print("Matplotlib ready / matplotlib 준비 완료")

## 1. Synthetic Sunspot Number Time Series / 합성 흑점수 시계열

**English.** We construct a synthetic sunspot time series using Hathaway, Wilson & Reichmann's (1994) cycle shape function (Eq. 6 in the review):
$$F(t) = A \left(\frac{t-t_0}{b}\right)^3 \left[\exp\left(\frac{t-t_0}{b}\right)^2 - c\right]^{-1}$$
with parameters A (amplitude, varied per cycle), b ≈ 56 months (fixed), c = 0.8, and t₀ = cycle start. Multiple cycles are concatenated with an 11-year base period, Gleissberg-like amplitude modulation (~100 yr), and additive rotation-scale noise.

**한국어.** Hathaway, Wilson & Reichmann(1994)의 주기 형태 함수(리뷰의 식 6)를 사용하여 합성 흑점 시계열을 만든다. 파라미터 A(진폭, 주기마다 변동), b ≈ 56개월(고정), c = 0.8, t₀ = 주기 시작. 11년 기본 주기, Gleissberg 유사 진폭 변조(~100년), 회전 척도의 가산 잡음을 포함한 여러 주기를 연결한다.

In [ ]:
def hwr_cycle(t, A, t0, b=56.0, c=0.8):
    """Hathaway-Wilson-Reichmann parametric cycle shape.

    Args:
        t: Time in months (array).
        A: Amplitude parameter.
        t0: Cycle start time in months.
        b: Rise time parameter in months (default 56).
        c: Asymmetry parameter (default 0.8).

    Returns:
        Sunspot number at each time in t (zero before t0).
    """
    dt = t - t0
    mask = dt > 0
    out = np.zeros_like(t, dtype=float)
    u = dt[mask] / b
    denom = np.exp(u * u) - c
    out[mask] = A * (u ** 3) / denom
    out[out < 0] = 0
    return out


def synthesize_ssn(n_cycles=24, dt_months=1.0):
    """Build a synthetic sunspot number time series spanning n_cycles.

    Each cycle has period drawn around 132 months (~11 yr), amplitude
    modulated by a Gleissberg-like 100-yr sinusoid plus random scatter.

    Args:
        n_cycles: Number of cycles to synthesize.
        dt_months: Sampling step in months.

    Returns:
        Tuple of (time_months, ssn, cycle_starts, cycle_amps).
    """
    cycle_starts = []
    cycle_amps = []
    t_start = 0.0
    for i in range(n_cycles):
        cycle_starts.append(t_start)
        gleissberg = 60.0 * np.sin(2 * np.pi * i / 9.1) ** 2 + 60.0
        amp = gleissberg + np.random.normal(0, 20)
        amp = max(amp, 25.0)
        cycle_amps.append(amp)
        period = np.random.normal(132, 8)
        t_start += period

    total_months = t_start + 150
    t = np.arange(0, total_months, dt_months)
    ssn = np.zeros_like(t)

    for t0, A in zip(cycle_starts, cycle_amps):
        f = hwr_cycle(t, A=1.0, t0=t0)
        if f.max() > 0:
            f = f * (A / f.max())
        ssn += f

    ssn += np.random.normal(0, 5, size=t.shape)
    ssn[ssn < 0] = 0
    return t, ssn, np.array(cycle_starts), np.array(cycle_amps)


t_mo, ssn, starts, amps = synthesize_ssn(n_cycles=24)
t_yr = t_mo / 12.0 + 1820

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_yr, ssn, lw=0.8, color='C0')
ax.set_xlabel("Year / 년")
ax.set_ylabel("Synthetic Sunspot Number / 합성 흑점수")
ax.set_title("Synthetic Solar Cycle Record (24 cycles) / 합성 태양 주기 기록")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean amplitude / 평균 진폭 = {amps.mean():.1f} (Hathaway: ~114)")
print(f"Amplitude std / 진폭 표준편차 = {amps.std():.1f} (Hathaway: ~40)")

## 2. Butterfly Diagram (Spörer's Law) / Butterfly Diagram (Spörer 법칙)

**English.** Per Eq. 8 in the review, active latitudes follow
$$\bar\lambda(t) = 28° \exp\left[-(t-t_0)/90\right]$$
with t in months since cycle start. We scatter synthetic sunspots in both hemispheres around this mean, with width σ_λ ~ 2–5° depending on cycle phase (Eq. 9).

**한국어.** 리뷰의 식 8에 따르면 활동 위도는 λ̄(t) = 28° exp[-(t-t₀)/90]을 따른다(t는 주기 시작 이후 개월). 양 반구에서 이 평균 주위로 흑점을 산포하며, 폭 σ_λ는 주기 위상에 따라 약 2–5°(식 9)이다.

In [ ]:
def sporer_latitude(t_since_start_months):
    """Mean active latitude vs. time since cycle start (Hathaway Eq. 8).

    Args:
        t_since_start_months: Time since cycle start in months.

    Returns:
        Mean absolute latitude in degrees.
    """
    return 28.0 * np.exp(-t_since_start_months / 90.0)


def zone_width(area):
    """Sunspot-zone RMS latitudinal width (Hathaway Eq. 9).

    Args:
        area: Total sunspot area in microhemispheres.

    Returns:
        Sigma_lambda in degrees.
    """
    return 1.5 + 3.8 * (1.0 - np.exp(-area / 400.0))


butterfly_t = []
butterfly_lat = []
for t0, A in zip(starts, amps):
    n_spots = int(A * 40)
    grid = np.arange(0, 140 * 12, 2) + t0
    env = hwr_cycle(grid, A=1.0, t0=t0)
    if env.sum() == 0:
        continue
    probs = env / env.sum()
    sampled_idx = np.random.choice(len(grid), size=n_spots, p=probs)
    t_spots = grid[sampled_idx]
    dt = t_spots - t0
    mean_lat = sporer_latitude(dt)
    area_proxy = 200 + 15 * A * env[sampled_idx] / env.max()
    width = zone_width(area_proxy)
    lat = np.random.normal(mean_lat, width)
    sign = np.random.choice([-1, 1], size=n_spots)
    lat = sign * np.abs(lat)
    butterfly_t.append(t_spots)
    butterfly_lat.append(lat)

bt = np.concatenate(butterfly_t) / 12.0 + 1820
bl = np.concatenate(butterfly_lat)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.scatter(bt, bl, s=0.5, alpha=0.3, color='C3')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel("Year / 년")
ax.set_ylabel("Latitude / 위도 (°)")
ax.set_title("Synthetic Butterfly Diagram — Sporer's Law\n\ud569성 Butterfly Diagram — Spörer 법칙")
ax.set_ylim(-40, 40)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Each wing drifts from ~25 deg to ~5 deg over a cycle, matching Hathaway Fig. 9 and Fig. 31.")
print("각 날개는 한 주기 동안 ~25도에서 ~5도로 이동하며 Hathaway 그림 9, 31과 일치.")

## 3. Waldmeier Effect / Waldmeier 효과

**English.** The review's Eq. 7: Rise Time (months) ≈ 35 + 1800 / Amplitude. Larger cycles rise faster. We measure rise times in our synthetic record and fit.

**한국어.** 리뷰의 식 7: 상승시간(개월) ≈ 35 + 1800/진폭. 큰 주기가 더 빠르게 상승. 합성 기록에서 상승시간을 측정하고 맞춤한다.

In [ ]:
def measure_rise_times(t_mo, ssn, cycle_starts):
    """Measure rise time and peak amplitude for each cycle.

    Args:
        t_mo: Time array in months.
        ssn: Sunspot number array.
        cycle_starts: Cycle start times in months.

    Returns:
        Tuple (amplitudes, rise_times) arrays.
    """
    rises, peaks = [], []
    cs = list(cycle_starts) + [t_mo[-1]]
    for i in range(len(cycle_starts)):
        mask = (t_mo >= cs[i]) & (t_mo < cs[i + 1])
        if mask.sum() < 10:
            continue
        segment = ssn[mask]
        tseg = t_mo[mask]
        peak_idx = np.argmax(segment)
        rise = tseg[peak_idx] - cs[i]
        if rise <= 5:
            continue
        peaks.append(segment[peak_idx])
        rises.append(rise)
    return np.array(peaks), np.array(rises)


peaks, rises = measure_rise_times(t_mo, ssn, starts)


def waldmeier(A, a=35, b=1800):
    """Waldmeier rise-time vs amplitude relation (Hathaway Eq. 7)."""
    return a + b / A


popt, _ = curve_fit(waldmeier, peaks, rises, p0=[35, 1800])
A_grid = np.linspace(30, 220, 200)

fig, ax = plt.subplots()
ax.scatter(peaks, rises, color='C0', s=60, label='Synthetic cycles / 합성 주기')
ax.plot(A_grid, waldmeier(A_grid, *popt),
        color='C3', lw=2,
        label=f'Fit: RT = {popt[0]:.0f} + {popt[1]:.0f}/A')
ax.plot(A_grid, waldmeier(A_grid, 35, 1800), '--', color='gray',
        label='Hathaway (Eq. 7): RT = 35 + 1800/A')
ax.set_xlabel("Cycle Amplitude / 주기 진폭")
ax.set_ylabel("Rise Time (months) / 상승시간 (개월)")
ax.set_title("Waldmeier Effect / Waldmeier 효과")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

r_corr = np.corrcoef(peaks, rises)[0, 1]
print(f"Correlation r = {r_corr:.3f} (Hathaway reports ~-0.7 in SSN data)")
print(f"상관계수 r = {r_corr:.3f} (Hathaway는 SSN 자료에서 약 -0.7 보고)")

## 4. Power Spectrum (Schwabe + Gleissberg) / 파워 스펙트럼

**English.** We compute a Welch power spectrum of the synthetic sunspot record. Expect peaks at the Schwabe frequency (~1/11 yr⁻¹) and at the Gleissberg modulation frequency (~1/100 yr⁻¹).

**한국어.** 합성 흑점 기록의 Welch 파워 스펙트럼을 계산한다. Schwabe 주파수(~1/11년⁻¹)와 Gleissberg 변조 주파수(~1/100년⁻¹)의 피크를 기대한다.

In [ ]:
years = t_mo / 12.0
ann_years = np.arange(years.min(), years.max(), 1.0)
ann_ssn = np.interp(ann_years, years, ssn)

fs = 1.0  # samples per year
f, pxx = welch(ann_ssn - ann_ssn.mean(), fs=fs, nperseg=min(128, len(ann_ssn)))
periods = 1.0 / np.where(f > 0, f, np.nan)

fig, ax = plt.subplots()
ax.loglog(periods, pxx, color='C0', lw=1.5)
ax.axvline(11, color='C3', linestyle='--', label='Schwabe ~11 yr')
ax.axvline(100, color='C2', linestyle='--', label='Gleissberg ~100 yr')
ax.axvline(22, color='C5', linestyle=':', alpha=0.6, label='Hale ~22 yr')
ax.set_xlabel("Period (years) / 주기 (년)")
ax.set_ylabel("PSD")
ax.set_title("Power Spectrum of Synthetic SSN / 합성 흑점수 파워 스펙트럼")
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

near_11 = (periods > 8) & (periods < 16)
dominant_period = periods[near_11][np.argmax(pxx[near_11])]
print(f"Dominant period near 11 yr: {dominant_period:.2f} yr")
print(f"11년 부근의 주 주기: {dominant_period:.2f}년")

## 5. Polar-Field Precursor Prediction / 극자기장 전조 예측

**English.** Following Schatten et al. (1978) and Svalgaard et al. (2005): the polar field at cycle minimum is the poloidal seed for the next cycle. In a Babcock–Leighton picture, the toroidal field (and hence the sunspot number) amplifies approximately linearly with the seed poloidal field. We build a synthetic polar-field time series from the butterfly poleward transport of following-polarity flux (Joy's Law implies tilt that leads to net poleward flux), then predict each cycle's amplitude from the polar field at the preceding minimum.

**한국어.** Schatten 등(1978)과 Svalgaard 등(2005)에 따라, 극소기의 극자기장은 다음 주기의 poloidal seed이다. Babcock–Leighton 관점에서, toroidal 자기장(따라서 흑점수)은 seed poloidal 자기장과 거의 선형적으로 증폭된다. Butterfly의 following-polarity 플럭스 극방향 수송으로부터 합성 극자기장 시계열을 구축하고, 직전 극소기의 극자기장으로 각 주기의 진폭을 예측한다.

In [ ]:
def synthesize_polar_field(t_mo, starts, amps):
    """Build a polar-field proxy from cycle amplitudes.

    The polar field rises during the declining phase and peaks near
    the next minimum, with a magnitude proportional to the sqrt of the
    declining-cycle amplitude (flux conservation argument).

    Args:
        t_mo: Time grid in months.
        starts: Cycle start times.
        amps: Cycle amplitudes.

    Returns:
        Polar-field proxy array (same shape as t_mo).
    """
    pf = np.zeros_like(t_mo, dtype=float)
    cs = list(starts) + [t_mo[-1] + 1]
    for i, (t0, A) in enumerate(zip(starts, amps)):
        t_next_min = cs[i + 1]
        peak_strength = 0.12 * np.sqrt(A)
        sigma = 24
        pf += peak_strength * np.exp(-((t_mo - t_next_min) ** 2) / (2 * sigma ** 2))
    for i, t0 in enumerate(starts):
        if i % 2 == 1:
            mask = (t_mo >= t0) & (t_mo < cs[i + 1])
            pf[mask] *= -1
    return pf


pf_series = synthesize_polar_field(t_mo, starts, amps)

pf_at_min = []
for i in range(len(starts) - 1):
    min_time = starts[i + 1]
    idx = np.argmin(np.abs(t_mo - min_time))
    pf_at_min.append(abs(pf_series[idx]))
pf_at_min = np.array(pf_at_min)
next_amps = np.array(amps[1:])

slope, intercept = np.polyfit(pf_at_min, next_amps, 1)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t_yr, pf_series, color='C2', lw=1.2, label='|Polar field proxy| / 극자기장 대용물')
axes[0].plot(t_yr, ssn / 10, color='C0', lw=0.7, alpha=0.6, label='SSN/10')
axes[0].set_xlabel("Year / 년")
axes[0].set_ylabel("Polar field (arb.) or SSN/10")
axes[0].set_title("Polar Field Proxy vs Sunspot Number / 극자기장 대용물 vs 흑점수")
axes[0].legend()
axes[0].grid(alpha=0.3)

pf_grid = np.linspace(pf_at_min.min(), pf_at_min.max(), 50)
axes[1].scatter(pf_at_min, next_amps, color='C3', s=60, label='Observed pairs / 관측 쌍')
axes[1].plot(pf_grid, slope * pf_grid + intercept, 'k--',
             label=f'Fit: A = {slope:.1f} PF + {intercept:.0f}')
axes[1].set_xlabel("|Polar Field| at Minimum / 극소기 극자기장 세기")
axes[1].set_ylabel("Next Cycle Amplitude / 다음 주기 진폭")
axes[1].set_title("Polar-field Precursor Method / 극자기장 전조 방법")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

r_pf = np.corrcoef(pf_at_min, next_amps)[0, 1]
print(f"Polar-field vs next-amplitude correlation r = {r_pf:.3f}")
print(f"극자기장 vs 다음 주기 진폭 상관계수 r = {r_pf:.3f}")

## 6. Babcock–Leighton Toy Flux-Transport Dynamo / Babcock–Leighton 장난감 flux-transport dynamo

**English.** A minimal 1D (time only) iterated map captures the essence of the flux-transport dynamo:
$$B_{tor,n+1} = \Omega \cdot B_{pol,n}, \qquad B_{pol,n+1} = -\alpha \cdot B_{tor,n+1} \cdot f(B_{tor,n+1})$$
where Ω (shear) and α (tilt–emergence) are coupling constants and f is a quenching function preventing runaway growth. The sign flip in the α term enforces Hale polarity reversal each cycle. Varying α with small stochastic perturbations generates amplitude variability resembling the real cycle.

**한국어.** 최소한의 1D(시간만) 반복 사상이 flux-transport dynamo의 본질을 포착한다: B_{tor,n+1} = Ω · B_{pol,n}, B_{pol,n+1} = -α · B_{tor,n+1} · f(B_{tor,n+1}). Ω(전단)와 α(기울기–출현)는 결합 상수이며, f는 폭주 성장을 막는 quenching 함수. α 항의 부호 반전은 매 주기 Hale 극성 반전을 실현. α에 작은 확률적 섭동을 가하면 실제 주기와 유사한 진폭 변동성이 생성된다.

In [ ]:
def bl_dynamo_map(n_cycles=40, omega=2.0, alpha0=0.35, B_sat=1.0, noise=0.12, seed=7):
    """Iterated Babcock-Leighton map for toroidal and poloidal fields.

    Args:
        n_cycles: Number of cycles to simulate.
        omega: Omega-effect coupling (shear).
        alpha0: Alpha-effect coupling (tilt/emergence).
        B_sat: Saturation magnetic field for quenching.
        noise: Fractional random perturbation on alpha per cycle.
        seed: RNG seed.

    Returns:
        Tuple (cycle_index, B_pol_series, B_tor_series).
    """
    rng = np.random.default_rng(seed)
    B_pol = np.zeros(n_cycles + 1)
    B_tor = np.zeros(n_cycles + 1)
    B_pol[0] = 0.5
    for n in range(n_cycles):
        alpha = alpha0 * (1.0 + noise * rng.standard_normal())
        B_tor[n + 1] = omega * B_pol[n]
        f_quench = 1.0 / (1.0 + (B_tor[n + 1] / B_sat) ** 2)
        B_pol[n + 1] = -alpha * B_tor[n + 1] * f_quench
    return np.arange(n_cycles + 1), B_pol, B_tor


idx, B_pol, B_tor = bl_dynamo_map(n_cycles=30, noise=0.15)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(idx, B_pol, 'o-', color='C2', label='Poloidal field $B_{pol}$')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel("$B_{pol}$")
axes[0].set_title("Toy Babcock-Leighton Flux-Transport Dynamo / 장난감 Babcock-Leighton dynamo")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(idx[1:], np.abs(B_tor[1:]), 's-', color='C3',
             label='|$B_{tor}$| (proxy for cycle amplitude)')
axes[1].set_xlabel("Cycle number / 주기 번호")
axes[1].set_ylabel("|$B_{tor}$|")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Observations / 관측:")
print("  Polar field flips sign every cycle (Hale) / 극자기장은 매 주기 부호 반전")
print(f"  Amplitude variability: std(|B_tor|)/mean = {np.std(np.abs(B_tor[1:])) / np.mean(np.abs(B_tor[1:])):.2%}")
print("  Hathaway reports cycle amplitude std / mean ~= 40/114 = 35%")

## 7. Summary and Connections / 요약과 연결

**English.** This notebook has reproduced, with synthetic data, the principal quantitative features of the solar cycle reviewed by Hathaway (2015):

- **Cycle shape**: HWR parametric form captures the fast-rise, slow-decay asymmetry.
- **Butterfly diagram**: Sporer's Law λ(t) = 28° exp[-(t-t₀)/90] reproduces the equatorward drift.
- **Waldmeier effect**: Rise time anti-correlates with amplitude, Rise ≈ 35 + 1800/A.
- **Power spectrum**: Peak near 11 yr plus a Gleissberg ~100-yr shoulder.
- **Polar-field precursor**: A polar-field proxy at minimum linearly predicts the next cycle amplitude — the physics underlying Schatten's and Svalgaard's method.
- **Babcock-Leighton dynamo toy model**: An iterated map with Omega- and alpha-effect couplings generates Hale polarity flips and cycle-to-cycle variability.

**한국어.** 이 노트북은 합성 자료로 Hathaway(2015)가 리뷰한 태양 주기의 주요 정량적 특성을 재현했다:

- **주기 형태**: HWR 파라메트릭 형태가 빠른 상승/느린 하강 비대칭을 포착.
- **Butterfly diagram**: Spörer 법칙 λ(t) = 28° exp[-(t-t₀)/90]이 적도 방향 이동을 재현.
- **Waldmeier 효과**: 상승시간이 진폭과 반상관, Rise ≈ 35 + 1800/A.
- **파워 스펙트럼**: 11년 부근 피크와 Gleissberg ~100년 shoulder.
- **극자기장 전조**: 극소기 극자기장 대용물이 다음 주기 진폭을 선형적으로 예측 — Schatten과 Svalgaard 방법의 근간 물리.
- **Babcock-Leighton dynamo 장난감 모델**: Omega-와 alpha-효과 결합을 가진 반복 사상이 Hale 극성 반전과 주기 간 변동성을 생성.

### Further reading / 추가 참고문헌

- Hathaway, D. H., *Living Reviews in Solar Physics*, **12**, 4 (2015) — the source of these implementations / 본 구현의 원천.
- Charbonneau, P., *Living Reviews in Solar Physics*, **7**, 3 (2010) — detailed dynamo theory / 상세 dynamo 이론.
- Svalgaard, L., Cliver, E. W., Kamide, Y., *GRL*, **32**, L01104 (2005) — polar field prediction / 극자기장 예측.
- Babcock, H. W., *ApJ*, **133**, 572 (1961) — foundational dynamo paper / 기초 dynamo 논문.